In [0]:
# CELL 1 — gold_conflict_of_interest
# Match officer names from Companies House against each other
# to find people who appear as directors at multiple companies
print("--- gold_conflict_of_interest ---")

from pyspark.sql import functions as F

DB = "fraud_analytics"

officers = spark.table(f"{DB}.silver_ch_officers")
overview = spark.table(f"{DB}.silver_ch_overview")

# Self-join officers — find same person at two different companies
conflicts = (officers.alias("a")
    .join(
        officers.alias("b"),
        (F.col("a.officer_name_clean") == F.col("b.officer_name_clean")) &
        (F.col("a.company_number")     != F.col("b.company_number"))
    )
    .join(
        overview.alias("ov_a"),
        F.col("a.company_number") == F.col("ov_a.company_number")
    )
    .join(
        overview.alias("ov_b"),
        F.col("b.company_number") == F.col("ov_b.company_number")
    )
    .select(
        F.col("a.officer_name_clean").alias("person_name"),
        F.col("a.company_number").alias("company_a_number"),
        F.col("ov_a.company_name").alias("company_a_name"),
        F.col("a.role").alias("role_at_company_a"),
        F.col("a.officer_status").alias("status_at_company_a"),
        F.col("b.company_number").alias("company_b_number"),
        F.col("ov_b.company_name").alias("company_b_name"),
        F.col("b.role").alias("role_at_company_b"),
        F.col("b.officer_status").alias("status_at_company_b"),
        F.col("a.dob_month").alias("dob_month"),
        F.col("a.dob_year").alias("dob_year"),
        F.current_timestamp().alias("gold_ts")
    )
    # Remove mirror duplicates (A-B and B-A are the same conflict)
    .filter(F.col("company_a_number") < F.col("company_b_number"))
    .dropDuplicates([
        "person_name", "company_a_number", "company_b_number"
    ])
    .orderBy("person_name")
)

(conflicts.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.gold_conflict_of_interest"))

count = conflicts.count()
print(f"Rows written: {count}")
conflicts.show(10, truncate=False)

In [0]:
# CELL 2 — gold_bank_anomalies
# Since our synthetic data doesn't have bank accounts yet,
# we build the framework ready for when that data lands
# For now we flag officers who resigned unusually quickly
print("--- gold_bank_anomalies ---")

from pyspark.sql import functions as F

officers = spark.table(f"{DB}.silver_ch_officers")
overview = spark.table(f"{DB}.silver_ch_overview")

# Flag officers who resigned within 6 months of appointment
# This is a governance red flag
short_tenure = (officers
    .filter(F.col("resigned_date").isNotNull())
    .filter(F.col("appointed_date").isNotNull())
    .withColumn("days_in_role",
        F.datediff(
            F.col("resigned_date"),
            F.col("appointed_date")
        )
    )
    .filter(F.col("days_in_role") <= 180)
    .join(overview, "company_number")
    .select(
        F.col("company_number"),
        F.col("company_name"),
        F.col("officer_name_clean").alias("officer_name"),
        F.col("role"),
        F.col("appointed_date"),
        F.col("resigned_date"),
        F.col("days_in_role"),
        F.lit("short_tenure").alias("flag_type"),
        F.lit("Officer resigned within 180 days").alias("flag_reason"),
        F.current_timestamp().alias("gold_ts")
    )
    .orderBy("days_in_role")
)

(short_tenure.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.gold_governance_flags"))

count = short_tenure.count()
print(f"Rows written: {count}")
short_tenure.show(10, truncate=False)

In [0]:
# CELL 3 — gold_market_signals (fixed)
print("--- gold_market_signals ---")

from pyspark.sql import functions as F
from pyspark.sql.window import Window

prices = spark.table(f"{DB}.silver_yf_prices")
stats  = spark.table(f"{DB}.silver_yf_stats")

w = (Window
     .partitionBy("ticker")
     .orderBy("trade_date")
     .rowsBetween(-30, 0))

signals = (prices
    .withColumn("rolling_avg",    F.avg("close").over(w))
    .withColumn("rolling_stddev", F.stddev("close").over(w))

    # try_divide returns NULL instead of crashing when stddev = 0
    .withColumn("z_score",
        F.expr(
            "try_divide(close - rolling_avg, rolling_stddev)"
        )
    )
    .filter(F.col("z_score").isNotNull())
    .withColumn("anomaly_flag",
        F.when(F.abs(F.col("z_score")) > 2, True)
         .otherwise(False)
    )
    .filter(F.col("anomaly_flag") == True)
    .join(
        stats.select("ticker", "sector", "industry", "market_cap"),
        "ticker"
    )
    .select(
        F.col("ticker"),
        F.col("company_name"),
        F.col("sector"),
        F.col("industry"),
        F.col("trade_date"),
        F.col("open"),
        F.col("close"),
        F.col("volume"),
        F.round("rolling_avg", 2).alias("rolling_avg_30d"),
        F.round("z_score", 4).alias("z_score"),
        F.when(F.col("z_score") > 2, "spike")
         .otherwise("crash").alias("signal_type"),
        F.current_timestamp().alias("gold_ts")
    )
    .orderBy("ticker", "trade_date")
)

(signals.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DB}.gold_market_signals"))

count = signals.count()
print(f"Rows written: {count}")
signals.show(10, truncate=False)

In [0]:
# CELL 4 — GOLD SUMMARY
print("\n" + "="*60)
print("  GOLD LAYER — FINAL TABLE SUMMARY")
print("="*60)

DB = "fraud_analytics"

gold_tables = [
    ("gold_conflict_of_interest", "People at multiple companies"),
    ("gold_governance_flags",     "Short tenure red flags"),
    ("gold_market_signals",       "Unusual price movements"),
]

for table, description in gold_tables:
    try:
        count = spark.table(f"{DB}.{table}").count()
        print(f"  {table:<32}  {count:>6,} rows  — {description}")
    except Exception as e:
        print(f"  {table:<32}  ERROR: {e}")

print("="*60)
print("\n  FULL MEDALLION ARCHITECTURE COMPLETE")
print("="*60)

all_tables = [
    "silver_ch_overview",
    "silver_ch_officers",
    "silver_ch_filings",
    "silver_yf_prices",
    "silver_yf_income",
    "silver_yf_balance",
    "silver_yf_cashflow",
    "silver_yf_stats",
    "gold_conflict_of_interest",
    "gold_governance_flags",
    "gold_market_signals",
]

total = 0
print(f"\n  {'Layer':<8}  {'Table':<32}  {'Rows':>8}")
print(f"  {'-'*56}")
for t in all_tables:
    layer = "GOLD" if t.startswith("gold") else "SILVER"
    try:
        count = spark.table(f"{DB}.{t}").count()
        total += count
        print(f"  {layer:<8}  {t:<32}  {count:>8,}")
    except Exception as e:
        print(f"  {layer:<8}  {t:<32}  ERROR")

print(f"  {'-'*56}")
print(f"  {'TOTAL':<8}  {'All tables':<32}  {total:>8,}")
print("="*60)